# 33. Do the pair-encoded models earn a place in the stack?

**One variable against ledger row 44** (`stack_logit_29_oof`, CV 0.967925): the member
set. Same logistic combiner, same `C`, same fold-wise protocol, same folds. Row 44 is
refit in this run rather than quoted.

## Why a null single model is not a null stack member

Rows 55 to 57 measured the two pair-encoded models as models and called them null.
`xgb_pair_top9` is +0.000077 over its own baseline and fails the gate; `xgb_pair_all66`
is -0.000509, five standard errors **below** it.

That is not the question a stack asks. This repo has already established, twice and
expensively, that a member's own CV is close to irrelevant to whether the combiner wants
it:

- The raw neural model scores **0.939169**, twenty-four thousandths behind the worst
  gradient booster here, and takes **+0.0912** of weight in the 24-member stack.
  Dropping it costs a tenth of the stack's gain.
- The whole diversity conclusion was withdrawn on 2026-08-12 for exactly this reason. It
  had been measured with an equal-weight combiner, where a weak member drags the average
  down. A fitted combiner does not average, it weights, and it can extract a direction
  from a member that is bad at everything else.

So the relevant statistic is not their CV. It is how much they disagree with what is
already in the stack.

| against `xgb_te` | Spearman |
|---|---|
| `xgb_pair_base` | 1.0000000 |
| `xgb_pair_top9` | 0.995332 |
| `xgb_pair_all66` | 0.993974 |

`xgb_pair_base` at exactly 1.0 is the reproduction check doing its job: same
configuration, bit-identical predictions, so it is excluded from the arms below rather
than added as a thirtieth copy of a member already present.

The other two are **more decorrelated from `xgb_te` than the XGBoost seeds are from each
other**, and they were produced by the same learner at the same seed. The only thing
that differs is the feature set, so the disagreement is the crossed columns talking.
That is the one argument for this experiment and it is worth one cheap run.

## Three arms, each one member set against row 44

| arm | members | what it adds |
|---|---|---|
| `29` | 29 | row 44 refit, the reproduction check |
| `30_top9` | 30 | `xgb_pair_top9` |
| `30_all66` | 30 | `xgb_pair_all66` |
| `31_both` | 31 | both, reported as a follow-up rather than as a headline |

`30_top9` and `30_all66` are each exactly one member against row 44. `31_both` changes
two things and is therefore reported as a combination, not as a one-variable result.

## The prediction, before the run

The stack has been in a **substitution regime** since row 39: a new member takes weight
from existing ones rather than adding to the total. `xgb_pair_top9` is the same learner
at the same seed with nine extra columns, which is the case where substitution should be
most complete.

**Prediction: both arms positive, both under the +0.00005 floor, with `30_all66` the
larger of the two** because it disagrees more. That would make this real-and-negligible,
the call row 32 got.

My last prediction, in `32`, was wrong. Recording this one anyway, because the point of
writing them down is the ones that miss.

## The gate, pre-registered

`27`'s stack gate, not the single-model gate: **at least 4 of 5 folds positive and at
least +0.00005 on the paired mean**, or this is logged as real-but-negligible rather
than as an improvement. A new best stack is only submitted if it clears that bar.


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold


def find_repo():
    for b in [Path.cwd(), *Path.cwd().parents]:
        if (b / "data" / "raw" / "train.csv").exists():
            return b
    raise FileNotFoundError("data/raw/train.csv not found")


REPO = find_repo()
O, S = REPO / "artifacts" / "oof", REPO / "submissions"

train = pd.read_csv(REPO / "data" / "raw" / "train.csv")
test = pd.read_csv(REPO / "data" / "raw" / "test.csv")
y = train["addicted_label"].to_numpy()

# The same split every vector on disk was produced under. Rebuilt rather than loaded,
# and then checked, because a silently different fold vector is the one error here
# that produces a clean-looking wrong answer.
folds = np.full(len(train), -1, dtype=np.int64)
for i, (_, va) in enumerate(StratifiedKFold(5, shuffle=True,
                                            random_state=42).split(train, y)):
    folds[va] = i
assert (folds >= 0).all() and np.bincount(folds).sum() == len(train)
print(f"train {len(train):,}  test {len(test):,}  folds {np.bincount(folds)}")

In [ ]:
# Row 44's twenty-nine, in row 44's order, then the two pair-encoded candidates.
MEM = [
    ("te42", O / "te_bag42_oof.npy", O / "te_bag42_test.npy"),
    ("te2024", O / "te_seed2024_oof.npy", O / "te_seed2024_test.npy"),
    ("te7", O / "te_seed7_oof.npy", O / "te_seed7_test.npy"),
    ("te2025", O / "te_seed2025_oof.npy", O / "te_seed2025_test.npy"),
    ("te13", O / "te_seed13_oof.npy", O / "te_seed13_test.npy"),
    ("anchor", O / "lgbm_default_anchor_seed42.npy",
     S / "lgbm_default_anchor_seed42.csv"),
    ("trees300", O / "lgbm_trees300_seed42.npy", S / "lgbm_trees300_seed42.csv"),
    ("trees1000", O / "lgbm_trees1000_seed42.npy", S / "lgbm_trees1000_seed42.csv"),
    ("trees2000", O / "lgbm_trees2000_seed42.npy", S / "lgbm_trees2000_seed42.csv"),
    ("lr010", O / "lgbm_lr01_n1000_seed42.npy", S / "lgbm_lr01_n1000_seed42.csv"),
    ("lr005", O / "lgbm_lr005_n2000_seed42.npy", S / "lgbm_lr005_n2000_seed42.csv"),
    ("lr003", O / "lgbm_lr003_n3333_seed42.npy", S / "lgbm_lr003_n3333_seed42.csv"),
    ("bag42", O / "lgbm_bag08_lr005_n2000_seed42.npy",
     S / "lgbm_bag08_lr005_n2000_seed42.csv"),
    ("bag2024", O / "lgbm_bag08_lr005_n2000_seed2024.npy",
     S / "lgbm_bag08_lr005_n2000_seed2024.csv"),
    ("bag7", O / "lgbm_bag08_lr005_n2000_seed7.npy",
     S / "lgbm_bag08_lr005_n2000_seed7.csv"),
    ("bag2025", O / "lgbm_bag08_lr005_n2000_seed2025.npy",
     S / "lgbm_bag08_lr005_n2000_seed2025.csv"),
    ("bag13", O / "lgbm_bag08_lr005_n2000_seed13.npy",
     S / "lgbm_bag08_lr005_n2000_seed13.csv"),
    ("neural", O / "neural_oof.npy", O / "neural_test.npy"),
    ("cat42", O / "catboost_te_oof.npy", O / "catboost_te_test.npy"),
    ("cat2024", O / "catboost_te_seed2024_oof.npy",
     O / "catboost_te_seed2024_test.npy"),
    ("cat7", O / "catboost_te_seed7_oof.npy", O / "catboost_te_seed7_test.npy"),
    ("cat2025", O / "catboost_te_seed2025_oof.npy",
     O / "catboost_te_seed2025_test.npy"),
    ("cat13", O / "catboost_te_seed13_oof.npy", O / "catboost_te_seed13_test.npy"),
    ("neural_te", O / "neural_te_oof.npy", O / "neural_te_test.npy"),
    ("xgb_te", O / "xgb_te_oof.npy", O / "xgb_te_test.npy"),
    ("xgb2024", O / "xgb_te_seed2024_oof.npy", O / "xgb_te_seed2024_test.npy"),
    ("xgb7", O / "xgb_te_seed7_oof.npy", O / "xgb_te_seed7_test.npy"),
    ("xgb2025", O / "xgb_te_seed2025_oof.npy", O / "xgb_te_seed2025_test.npy"),
    ("xgb13", O / "xgb_te_seed13_oof.npy", O / "xgb_te_seed13_test.npy"),
    ("pair_top9", O / "xgb_pair_top9_oof.npy", O / "xgb_pair_top9_test.npy"),
    ("pair_all66", O / "xgb_pair_all66_oof.npy", O / "xgb_pair_all66_test.npy"),
]
CAND = ["pair_top9", "pair_all66"]
XG = ["xgb_te", "xgb2024", "xgb7", "xgb2025", "xgb13"]


def logit(p):
    p = np.clip(np.asarray(p, dtype=float), 1e-9, 1 - 1e-9)
    return np.clip(np.log(p / (1 - p)), -30, 30)


def load_test(path):
    if path.suffix == ".npy":
        return np.load(path)
    df = pd.read_csv(path)
    # A csv written in a different row order would blend perfectly cleanly and be
    # undetectable in the score. Checked rather than assumed.
    assert (df["id"].to_numpy() == test["id"].to_numpy()).all(), f"id order {path.name}"
    return df["addicted_label"].to_numpy()


names = [m[0] for m in MEM]
Poof = {n: np.load(p) for n, p, _ in MEM}
Ptest = {n: load_test(t) for n, _, t in MEM}

for n in names:
    assert Poof[n].shape == (len(train),), n
    assert Ptest[n].shape == (len(test),), n
    # A partially failed run leaves a constant fold, which blends silently.
    assert min(np.ptp(Poof[n][folds == f]) for f in range(5)) > 0, f"dead fold in {n}"

Loof = np.column_stack([logit(Poof[n]) for n in names])
Ltest = np.column_stack([logit(Ptest[n]) for n in names])

BASE29 = [i for i, n in enumerate(names) if n not in CAND]
IDX = {n: i for i, n in enumerate(names)}
ARMS = {
    "29": BASE29,
    "30_top9": BASE29 + [IDX["pair_top9"]],
    "30_all66": BASE29 + [IDX["pair_all66"]],
    "31_both": BASE29 + [IDX["pair_top9"], IDX["pair_all66"]],
}
print(f"{len(names)} vectors loaded, oof {Loof.shape}, test {Ltest.shape}")
print("candidate CV, and their disagreement with what is already in the stack:")
for n in CAND:
    cv = np.mean([roc_auc_score(y[folds == f], Poof[n][folds == f]) for f in range(5)])
    rho = pd.Series(Poof[n]).corr(pd.Series(Poof["xgb_te"]), method="spearman")
    print(f"  {n:11} CV {cv:.6f}   spearman vs xgb_te {rho:.6f}")


In [ ]:
def run(cols):
    oof = np.zeros(len(train))
    tst = np.zeros((5, len(test)))
    cf = np.zeros((5, len(cols)))
    for f in range(5):
        tr, va = folds != f, folds == f
        clf = LogisticRegression(C=1.0, max_iter=2000).fit(Loof[np.ix_(tr, cols)],
                                                           y[tr])
        oof[va] = clf.decision_function(Loof[np.ix_(va, cols)])
        tst[f] = clf.decision_function(Ltest[:, cols])
        cf[f] = clf.coef_[0]
    per = np.array([roc_auc_score(y[folds == f], oof[folds == f]) for f in range(5)])
    return per, tst, cf


res = {a: run(cols) for a, cols in ARMS.items()}
per = {a: r[0] for a, r in res.items()}

ROW44_CV = 0.967925
repro = per["29"].mean() - ROW44_CV
REPRODUCED = abs(repro) < 1e-4

print(f"{'':12} {'fold 0':>9} {'fold 1':>9} {'fold 2':>9} {'fold 3':>9} {'fold 4':>9}"
      f" {'mean':>10}")
for a in ARMS:
    print(f"{a:12} " + " ".join(f"{v:9.6f}" for v in per[a]) + f" {per[a].mean():10.6f}")
print()
print(f"29-member CV {per['29'].mean():.6f} +/- {per['29'].std():.6f}"
      f"   (row 44 recorded {ROW44_CV:.6f}, diff {repro:+.2e})")
if not REPRODUCED:
    print("\nROW 44 DID NOT REPRODUCE. Nothing below is comparable to it.")


In [ ]:
def paired(a, b, lbl):
    d = a - b
    t = d.mean() / (d.std(ddof=1) / np.sqrt(len(d)))
    print(f"{lbl:34} {d.mean():+.6f}  sd {d.std(ddof=1):.6f}  "
          f"{(d > 0).sum()}/5  t(4)={t:.2f}")
    print("     per fold: " + "  ".join(f"{v:+.6f}" for v in d))
    return d


D = {a: paired(per[a], per["29"], f"{a} vs 29 (row 44)")
     for a in ARMS if a != "29"}

GATE_FLOOR = 5e-05
print()
print("For scale, every addition this stack has accepted:")
print("  18 -> 19, CatBoost          +0.000100  (row 27)")
print("  19 -> 23, four cat seeds    +0.000014  (row 32)")
print("  23 -> 24, neural_te         +0.000043  (row 34)")
print("  24 -> 25, XGBoost           +0.000066  (row 39)")
print("  25 -> 29, four xgb seeds    +0.000052  (row 44)")
print("The prediction in the header: both positive, both under the floor,")
print("  30_all66 larger than 30_top9.")

print()
if not REPRODUCED:
    print("VERDICT: blocked, row 44 did not reproduce")
else:
    best = max((a for a in D), key=lambda a: D[a].mean())
    d, wins, mean_g = D[best], int((D[best] > 0).sum()), float(D[best].mean())
    if wins >= 4 and mean_g >= GATE_FLOOR:
        print(f"VERDICT: {best} earns its place on the pre-registered gate at "
              f"{mean_g:+.6f}.")
        print("  This is a new best stack and it is worth a submission slot.")
    elif wins >= 4 and mean_g > 0:
        print(f"VERDICT: real and negligible. {best} is {mean_g:+.6f} at {wins}/5,")
        print("  positive but under the floor, which is the call row 32 got.")
    else:
        print("VERDICT: the pair-encoded models add nothing the stack wants.")
        print("  A null as a model and a null as a member, which is the stronger")
        print("  version of rows 55 to 57 rather than a repeat of them.")


In [ ]:
# Where does any new weight come from? The stack has been substituting rather than
# adding since row 39, and the five XGBoost members are the obvious donors.
c29 = {names[i]: res["29"][2][:, i].mean() for i in ARMS["29"]}
best_arm = max((a for a in ARMS if a != "29"),
               key=lambda a: per[a].mean())
cB = {names[c]: res[best_arm][2][:, k].mean()
      for k, c in enumerate(ARMS[best_arm])}

print(f"largest movers, 29 members -> {best_arm}")
print(f"{'member':12} {'29':>10} {best_arm:>12} {'change':>10}")
moved = sorted(((n, c29.get(n), cB[n]) for n in cB),
               key=lambda r: -abs(r[2] - (r[1] if r[1] is not None else 0.0)))
for n, a, b in moved[:10]:
    if a is None:
        print(f"{n:12} {'new':>10} {b:>+12.4f} {'':>10}")
    else:
        print(f"{n:12} {a:>+10.4f} {b:>+12.4f} {b - a:>+10.4f}")

xg29 = sum(c29[n] for n in XG)
xgB = sum(cB[n] for n in XG)
print()
print(f"sum of the 5 XGBoost coefficients, 29 members : {xg29:+.4f}")
print(f"the same five in {best_arm:<22}: {xgB:+.4f}")
print(f"change                                        : {xgB - xg29:+.4f}")
newc = sum(cB[n] for n in CAND if n in cB)
print(f"weight taken by the new member(s)             : {newc:+.4f}")
print()
print("Substitution would show the XGBoost sum falling by about what the new member")
print("takes. Addition would show it roughly unchanged.")


In [ ]:
# The gate decides the VERDICT. It does not decide membership, and it does not decide
# whether a csv is worth writing. Bundling all three into one condition was an error in
# the first version of this cell, and the ledger already shows why:
#
#   row 34 added neural_te at +0.000043, UNDER this same floor, at 5/5 folds. It was
#   kept as a member and it is in row 44's twenty-nine today. Row 32 added four
#   CatBoost seeds at +0.000014, also kept.
#
# So the established practice is that a sub-floor addition stays in the member set and
# is logged as real-and-negligible. The rule below is written to match what this repo
# actually does, and the verdict above is unchanged: still negligible, still not an
# improvement.
#
# A submission is written when the arm is CV-best and positive on all five folds. The
# reason is not that it will move the leaderboard, which at +0.000043 against a public
# LB standard error near 0.001 it cannot. It is that the CV-to-LB tracking record is
# itself a deliverable here, this is a free tenth observation for it, and the daily
# quota is 10 rather than the scarce resource three earlier rows were held back for.
best = max((a for a in ARMS if a != "29"), key=lambda a: D[a].mean())
wins, mean_g = int((D[best] > 0).sum()), float(D[best].mean())
CLEARED = REPRODUCED and wins >= 4 and mean_g >= GATE_FLOOR
WRITE = REPRODUCED and wins == 5 and per[best].mean() > per["29"].mean()

print(f"gate cleared (an improvement): {CLEARED}")
print(f"write a submission           : {WRITE}")
print()

if WRITE:
    pred = res[best][1].mean(axis=0)
    prob = 1 / (1 + np.exp(-pred))
    prev = pd.read_csv(S / "stack_oof_29.csv")
    assert (prev["id"].to_numpy() == test["id"].to_numpy()).all()
    rho = pd.Series(pred).corr(
        pd.Series(prev["addicted_label"].to_numpy()), method="spearman")
    out = S / f"stack_pair_{best}.csv"
    sub = pd.DataFrame({"id": test["id"], "addicted_label": prob})
    assert len(sub) == len(test) and sub["addicted_label"].between(0, 1).all()
    sub.to_csv(out, index=False)
    print(f"spearman vs row 44 on disk: {rho:.7f}")
    print(f"wrote {out.name}, {len(sub):,} rows")
else:
    print("not CV-best or not 5/5, so no submission written.")

print()
print("ledger lines:")
for a in ARMS:
    if a == "29":
        print(f"  stack_{a:<12} cv_mean {per[a].mean():.6f}  "
              f"cv_std {per[a].std():.6f}   (row 44 refit)")
    else:
        print(f"  stack_{a:<12} cv_mean {per[a].mean():.6f}  "
              f"cv_std {per[a].std():.6f}   vs 29 {D[a].mean():+.6f} "
              f"({int((D[a] > 0).sum())}/5)")
